# Notebook 09s: DINOv2 ViT-L/14 TransReID Training on CityFlowV2

This notebook trains a DINOv2 ViT-L/14 TransReID vehicle ReID model on CityFlowV2 for Kaggle P100 GPUs.

Key points:
- DINOv2 ViT-L/14 backbone with TransReID-style SIE + JPM + BNNeck
- CityFlowV2 crop extraction and train/query/gallery split based on the working 09q recipe
- AdamW with layer-wise LR decay, warmup, cosine decay, ID + triplet + center loss
- Best checkpoint tracked by training accuracy, with optional query/gallery evaluation when a split is available

In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "timm>=0.9", "einops", "gdown"
], check=True)

In [ ]:
import sys as _sys

class _Tee:
    def __init__(self, file_path):
        self._log = open(file_path, 'w', buffering=1, encoding='utf-8')
        self._stdout = _sys.stdout
    def write(self, msg):
        self._stdout.write(msg)
        self._log.write(msg)
    def flush(self):
        self._stdout.flush()
        self._log.flush()
    def isatty(self):
        return False

_LOG_PATH = '/kaggle/working/run_log.txt'
_tee = _Tee(_LOG_PATH)
_sys.stdout = _tee
_sys.stderr = _tee  # also capture tracebacks
print(f'[LOG] Logging stdout+stderr to {_LOG_PATH}')

import json
import math
import os
import random
import shutil
import time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
	props = torch.cuda.get_device_properties(0)
	print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {props.total_memory / 1024**3:.1f} GB")
print(f"timm: {timm.__version__}")

In [ ]:
# -- Download CityFlowV2 from Google Drive --
# All large files go to /tmp/ to avoid filling /kaggle/working/ (~20GB limit).
import subprocess, zipfile, re as _re

CITYFLOW_DIR = Path("/tmp/cityflowv2")
GDRIVE_ID = "13wNJpS_Oaoe-7y5Dzexg_Ol7bKu1OWuC"
ARCHIVE_NAME = "AIC22_Track1_MTMC_Tracking.zip"

# CityFlowV2 is an MTMC benchmark: vehicle IDs in gt.txt are GLOBALLY consistent
# across all 46 cameras / all splits. 880 distinct vehicle identities total.
# We use train + validation (have GT); test (S06) has no GT.
ALLOWED_SPLITS = {"train", "validation"}

# Check if already available (e.g. from previous run or attached dataset)
already_found = False
for check_dir in [CITYFLOW_DIR, Path("/kaggle/input/cityflowv2"), Path("/kaggle/input/aic22-track1-mtmc-tracking")]:
    if check_dir.exists() and any(list(check_dir.rglob("vdo.avi"))[:1]):
        print(f"CityFlowV2 already available at {check_dir}")
        CITYFLOW_DIR = check_dir
        already_found = True
        break

if not already_found:
    CITYFLOW_DIR.mkdir(parents=True, exist_ok=True)
    archive_path = Path(f"/tmp/{ARCHIVE_NAME}")

    if not archive_path.exists():
        print(f"Downloading CityFlowV2 from Google Drive (id={GDRIVE_ID})...")
        print("This may take 10-20 minutes for the full dataset (~20GB).")
        import gdown
        gdown.download(
            f"https://drive.google.com/uc?id={GDRIVE_ID}",
            str(archive_path), quiet=False
        )
    else:
        print(f"Archive already downloaded: {archive_path}")

    # Extract to a staging dir to avoid polluting /tmp/
    staging = Path("/tmp/_aic22_staging")
    staging.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {archive_path} to {staging}...")
    with zipfile.ZipFile(str(archive_path), "r") as zf:
        zf.extractall(str(staging))

    # Delete archive immediately to reclaim ~20GB
    if archive_path.exists():
        archive_path.unlink()
        print("Deleted archive to reclaim space.")

    # Debug: show what was extracted
    print("\nExtracted contents of staging dir:")
    for item in sorted(staging.iterdir()):
        print(f"  {item.name} ({'dir' if item.is_dir() else 'file'})")

    # Find all vdo.avi files under staging and flatten into CITYFLOW_DIR
    # Keep train + validation (both have GT); skip test (S06, no GT).
    moved = 0
    skipped_splits = set()
    for vdo_path in sorted(staging.rglob("vdo.avi")):
        cam_dir = vdo_path.parent       # e.g. .../c001/
        scene_dir = cam_dir.parent      # e.g. .../S01/
        split_dir = scene_dir.parent    # e.g. .../train/
        cam_name = cam_dir.name         # e.g. c001
        scene_name = scene_dir.name     # e.g. S01
        split_name = split_dir.name     # e.g. train
        # Only use dirs that look like camera/scene names
        if not cam_name.startswith("c") or not scene_name.startswith("S"):
            print(f"  SKIP unusual path: {vdo_path}")
            continue
        if split_name not in ALLOWED_SPLITS:
            skipped_splits.add(split_name)
            print(f"  SKIP {scene_name}_{cam_name}: {split_name} split (no GT)")
            continue
        flat_name = f"{scene_name}_{cam_name}"  # e.g. S01_c001
        target = CITYFLOW_DIR / flat_name
        if not target.exists():
            shutil.move(str(cam_dir), str(target))
            moved += 1
            if moved <= 15:
                print(f"  {flat_name} -> {target}")
            elif moved == 16:
                print(f"  ... (showing first 15 only)")
    print(f"Total cameras moved: {moved}")
    if skipped_splits:
        print(f"Skipped splits: {sorted(skipped_splits)} (no GT annotations)")

    # Also check if the zip extracted directly into /tmp/ (no wrapper dir)
    if moved == 0:
        print("Trying fallback: zip may have extracted directly to /tmp/...")
        for split_name in sorted(ALLOWED_SPLITS):
            split_dir = Path(f"/tmp/{split_name}")
            if not split_dir.exists():
                continue
            for vdo_path in sorted(split_dir.rglob("vdo.avi")):
                cam_dir = vdo_path.parent
                scene_dir = cam_dir.parent
                cam_name = cam_dir.name
                scene_name = scene_dir.name
                if not cam_name.startswith("c") or not scene_name.startswith("S"):
                    continue
                flat_name = f"{scene_name}_{cam_name}"
                target = CITYFLOW_DIR / flat_name
                if not target.exists():
                    shutil.move(str(cam_dir), str(target))
                    moved += 1
                    if moved <= 15:
                        print(f"  {flat_name} -> {target}")
        print(f"Fallback cameras moved: {moved}")

    # Cleanup staging + leftover split dirs
    if staging.exists():
        shutil.rmtree(str(staging), ignore_errors=True)
    for d in [Path("/tmp/train"), Path("/tmp/test"), Path("/tmp/validation")]:
        if d.exists():
            shutil.rmtree(str(d), ignore_errors=True)
    # Clean other extracted files
    for f in [Path("/tmp/ReadMe.txt"), Path("/tmp/list_cam.txt")]:
        if f.exists():
            f.unlink()
    for f in Path("/tmp").glob("Dataset License*"):
        f.unlink(missing_ok=True)
    for d in [Path("/tmp/cam_framenum"), Path("/tmp/cam_loc"), Path("/tmp/cam_timestamp"), Path("/tmp/eval")]:
        if d.exists():
            shutil.rmtree(str(d), ignore_errors=True)

# Verify -- only count dirs matching SXX_cXXX pattern
_cam_re = _re.compile(r"^S\d{2}_c\d{3}$")
found_cams = sorted([d.name for d in CITYFLOW_DIR.iterdir() if d.is_dir() and _cam_re.match(d.name)]) if CITYFLOW_DIR.exists() else []
print(f"\nCityFlowV2 at {CITYFLOW_DIR}")
print(f"Camera dirs found: {len(found_cams)} (46 unique physical cameras)")
print(f"  {found_cams}")
if not found_cams:
    print("\nERROR: No cameras found! Listing /tmp/ for debugging:")
    for p in sorted(Path("/tmp").iterdir()):
        if not p.name.startswith("uv-") and not p.name.startswith("tmp"):
            print(f"  {p.name} ({'dir' if p.is_dir() else 'file'})")

In [ ]:
# DATA_ROOT is detected dynamically after CityFlowV2 download (see previous cell)
# CITYFLOW_DIR is set by the download cell above
WEIGHTS_DIR = Path("/kaggle/input/mtmc-weights")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CROP_DIR = Path("/tmp/cityflowv2_dinov2_crops")
CROP_DIR.mkdir(parents=True, exist_ok=True)

VIT_MODEL = "vit_large_patch14_dinov2"
IMG_SIZE = 252
STRIDE_SIZE = 14

BATCH_SIZE = 32
NUM_EPOCHS = 120
BACKBONE_LR = 1.5e-5
HEAD_LR = 1.5e-4
WARMUP_EPOCHS = 10
WEIGHT_DECAY = 1e-4
MARGIN = 0.3
CENTER_LOSS_WEIGHT = 0.0005
CENTER_START = 10  # start center loss after warmup

PIDS_PER_BATCH = 8
INSTANCES_PER_PID = 4
MAX_CROPS_PER_ID_CAM = 20
MIN_AREA = 2000
MIN_BBOX_SIDE = 30
TRAIN_RATIO = 0.7
MIN_CAMS_FOR_EVAL = 2
EVAL_CHUNK_SIZE = 1024
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if BATCH_SIZE != PIDS_PER_BATCH * INSTANCES_PER_PID:
    raise ValueError("BATCH_SIZE must equal PIDS_PER_BATCH * INSTANCES_PER_PID for PK sampling")

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

BEST_MODEL_PATH = OUTPUT_DIR / "vehicle_transreid_dinov2_large_cityflowv2_best.pth"
FINAL_MODEL_PATH = OUTPUT_DIR / "vehicle_transreid_dinov2_large_cityflowv2_final.pth"
SUMMARY_PATH = OUTPUT_DIR / "vehicle_transreid_dinov2_large_cityflowv2_summary.json"

print(f"Device: {DEVICE}")
print(f"Training epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE} ({PIDS_PER_BATCH} identities x {INSTANCES_PER_PID} instances)")
print(f"CityFlowV2 download dir: {CITYFLOW_DIR}")
print(f"Weights dir available: {WEIGHTS_DIR.exists()}")

In [ ]:
# -- Locate CityFlowV2 data --
# Use ALL cameras with GT (no target subset filtering)
TARGET_CAMS = set()  # empty = use all available cameras
MAX_CROPS_PER_ID_CAM = 20  # more crops per ID for larger dataset
MIN_AREA = 2000
MIN_BBOX_SIDE = 30
TRAIN_RATIO = 0.7
MIN_CAMS_FOR_EVAL = 2
SEED = 42

# DATA_ROOT is directory containing S0X_c0XX/ subdirs
possible_roots = [
    CITYFLOW_DIR,
    Path("/tmp/cityflowv2"),
    Path("/kaggle/input/cityflowv2"),
    Path("/kaggle/input/aic22-track1-mtmc-tracking"),
    Path("/tmp/data/raw/cityflowv2"),
    Path("data/raw/cityflowv2"),
]

DATA_ROOT = None
for root in possible_roots:
    if root.exists():
        # Check for any camera dir pattern S0X_c0XX
        cam_dirs = [d for d in root.iterdir() if d.is_dir() and "_c0" in d.name]
        if cam_dirs:
            DATA_ROOT = root
            break
    if DATA_ROOT:
        break

if DATA_ROOT is None:
    # Search more broadly
    for base in [Path("/tmp"), Path("/kaggle/input")]:
        if not base.exists():
            continue
        vdos = list(base.rglob("vdo.avi"))[:1]
        if vdos:
            # Go up to the parent containing S0X dirs
            cam = vdos[0].parent
            DATA_ROOT = cam.parent.parent if cam.parent.name.startswith("S0") else cam.parent
            break

if DATA_ROOT is None:
    raise RuntimeError(
        "CityFlowV2 not found. Either:\n"
        "  1. Attach a CityFlowV2 dataset on Kaggle\n"
        "  2. Check the download cell output for errors\n"
        "  3. Locally: python scripts/download_datasets.py --dataset cityflowv2"
    )

print(f"CityFlowV2 data root: {DATA_ROOT}")

# Find available cameras with GT + video
def find_gt(cam_dir):
    """Find gt.txt -- could be at gt.txt or gt/gt.txt"""
    direct = cam_dir / "gt.txt"
    if direct.exists():
        return direct
    nested = cam_dir / "gt" / "gt.txt"
    if nested.exists():
        return nested
    # Search for any gt.txt
    matches = list(cam_dir.rglob("gt.txt"))
    return matches[0] if matches else None

def find_video(cam_dir):
    """Find video file"""
    for ext in ["avi", "mp4"]:
        for name in ["vdo", "video"]:
            p = cam_dir / f"{name}.{ext}"
            if p.exists():
                return p
    vids = list(cam_dir.glob("*.avi")) + list(cam_dir.glob("*.mp4"))
    return vids[0] if vids else None

CAMERAS = []
cam_gt_paths = {}
cam_vid_paths = {}

for cam_dir in sorted(DATA_ROOT.iterdir()):
    if not cam_dir.is_dir():
        continue
    gt = find_gt(cam_dir)
    vid = find_video(cam_dir)
    cam_name = cam_dir.name
    if gt and vid:
        CAMERAS.append(cam_name)
        cam_gt_paths[cam_name] = gt
        cam_vid_paths[cam_name] = vid
        print(f"  OK {cam_name}: GT={gt.name} Video={vid.name}")
    else:
        missing = []
        if not gt: missing.append("GT")
        if not vid: missing.append("video")
        print(f"  SKIP {cam_name}: missing {', '.join(missing)}")

if not CAMERAS:
    raise RuntimeError("No cameras with both GT and video found!")

print(f"\nUsing {len(CAMERAS)}/{len(found_cams)} cameras: {CAMERAS}")

In [ ]:
CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD = [0.26862954, 0.26130258, 0.27577711]

def find_gt(cam_dir: Path) -> Path | None:
    direct = cam_dir / "gt.txt"
    if direct.exists():
        return direct
    nested = cam_dir / "gt" / "gt.txt"
    if nested.exists():
        return nested
    matches = list(cam_dir.rglob("gt.txt"))
    return matches[0] if matches else None

def find_video(cam_dir: Path) -> Path | None:
    for name in ("vdo.avi", "video.avi", "vdo.mp4", "video.mp4"):
        candidate = cam_dir / name
        if candidate.exists():
            return candidate
    matches = list(cam_dir.glob("*.avi")) + list(cam_dir.glob("*.mp4"))
    return matches[0] if matches else None

camera_specs = []
flat_camera_dirs = [cam_dir for cam_dir in sorted(DATA_ROOT.iterdir()) if cam_dir.is_dir() and "_c" in cam_dir.name]
if flat_camera_dirs:
    for cam_dir in flat_camera_dirs:
        gt_path = find_gt(cam_dir)
        video_path = find_video(cam_dir)
        if gt_path is None or video_path is None:
            continue
        cam_name = cam_dir.name
        camera_specs.append((cam_name, gt_path, video_path))
else:
    dataset_root = DATA_ROOT.parent if DATA_ROOT.name == "train" else DATA_ROOT
    split_roots = [DATA_ROOT]
    validation_root = dataset_root / "validation"
    if validation_root.exists():
        split_roots.append(validation_root)

    for split_root in split_roots:
        if not split_root.exists():
            continue
        for scene_dir in sorted(split_root.iterdir()):
            if not scene_dir.is_dir():
                continue
            for cam_dir in sorted(scene_dir.iterdir()):
                if not cam_dir.is_dir():
                    continue
                gt_path = find_gt(cam_dir)
                video_path = find_video(cam_dir)
                if gt_path is None or video_path is None:
                    continue
                cam_name = f"{scene_dir.name}_{cam_dir.name}"
                camera_specs.append((cam_name, gt_path, video_path))

if not camera_specs:
    raise RuntimeError(f"No CityFlowV2 cameras found under {DATA_ROOT}")

print(f"Using {len(camera_specs)} camera directories")
for cam_name, gt_path, video_path in camera_specs[:8]:
    print(f"  {cam_name}: GT={gt_path.name} | Video={video_path.name}")

def load_gt(gt_path: Path) -> list[tuple[int, int, int, int, int, int]]:
    rows = []
    with gt_path.open("r", encoding="utf-8") as handle:
        for raw_line in handle:
            parts = raw_line.strip().split(",")
            if len(parts) < 6:
                continue
            frame, tid = int(parts[0]), int(parts[1])
            x, y, w, h = int(parts[2]), int(parts[3]), int(parts[4]), int(parts[5])
            rows.append((frame, tid, x, y, w, h))
    return rows

def extract_crops_from_camera(
    cam_name: str,
    gt_path: Path,
    video_path: Path,
    crop_dir: Path,
    max_crops: int,
    min_area: int,
) -> dict[int, list[str]]:
    gt_rows = load_gt(gt_path)
    if not gt_rows:
        return {}

    id_to_detections = defaultdict(list)
    for frame, tid, x, y, w, h in gt_rows:
        id_to_detections[tid].append((frame, x, y, w, h))

    frame_to_detections = defaultdict(list)
    for tid, detections in id_to_detections.items():
        if len(detections) <= max_crops:
            sampled = detections
        else:
            step = len(detections) / max_crops
            sampled = [detections[int(i * step)] for i in range(max_crops)]
        for frame, x, y, w, h in sampled:
            if w * h >= min_area and w >= MIN_BBOX_SIDE and h >= MIN_BBOX_SIDE:
                frame_to_detections[frame].append((tid, x, y, w, h))

    if not frame_to_detections:
        return {}

    crops = defaultdict(list)
    cap = cv2.VideoCapture(str(video_path))
    current_frame = 0
    target_frames = sorted(frame_to_detections.keys())
    target_index = 0

    while target_index < len(target_frames):
        ok, image = cap.read()
        if not ok:
            break
        current_frame += 1
        if current_frame < target_frames[target_index]:
            continue
        if current_frame > target_frames[target_index]:
            while target_index < len(target_frames) and target_frames[target_index] < current_frame:
                target_index += 1
            if target_index >= len(target_frames) or current_frame != target_frames[target_index]:
                continue

        height, width = image.shape[:2]
        for tid, x, y, w, h in frame_to_detections[current_frame]:
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(width, x + w), min(height, y + h)
            if x2 - x1 < MIN_BBOX_SIDE or y2 - y1 < MIN_BBOX_SIDE:
                continue
            crop = image[y1:y2, x1:x2]
            filename = f"{tid:04d}_{cam_name}_f{current_frame:06d}.jpg"
            destination = crop_dir / filename
            cv2.imwrite(str(destination), crop)
            crops[tid].append(str(destination))
        target_index += 1

    cap.release()
    return dict(crops)

existing_crops = list(CROP_DIR.glob("*.jpg"))
all_crops = defaultdict(lambda: defaultdict(list))
if len(existing_crops) > 100:
    print(f"Reusing {len(existing_crops)} existing crops from {CROP_DIR}")
    for crop_path in sorted(existing_crops):
        parts = crop_path.stem.split("_")
        tid = int(parts[0])
        cam_name = f"{parts[1]}_{parts[2]}"
        all_crops[tid][cam_name].append(str(crop_path))
else:
    print("Extracting crops from CityFlowV2 videos...")
    for cam_name, gt_path, video_path in camera_specs:
        cam_crops = extract_crops_from_camera(
            cam_name=cam_name,
            gt_path=gt_path,
            video_path=video_path,
            crop_dir=CROP_DIR,
            max_crops=MAX_CROPS_PER_ID_CAM,
            min_area=MIN_AREA,
        )
        for tid, paths in cam_crops.items():
            all_crops[tid][cam_name].extend(paths)

all_crops = {tid: dict(cam_map) for tid, cam_map in all_crops.items()}
total_crops = sum(sum(len(paths) for paths in cam_map.values()) for cam_map in all_crops.values())
print(f"Total crops: {total_crops}")
print(f"Vehicle IDs: {len(all_crops)}")

if not all_crops:
    raise RuntimeError("No crops extracted from CityFlowV2")

rng = np.random.RandomState(SEED)
multi_cam_ids = sorted(tid for tid, cams in all_crops.items() if len(cams) >= MIN_CAMS_FOR_EVAL)
single_cam_ids = sorted(tid for tid, cams in all_crops.items() if len(cams) < MIN_CAMS_FOR_EVAL)

if not multi_cam_ids:
    all_ids = sorted(all_crops.keys())
    rng.shuffle(all_ids)
    multi_cam_ids = all_ids
    single_cam_ids = []

rng.shuffle(multi_cam_ids)
n_train = max(int(len(multi_cam_ids) * TRAIN_RATIO), 1)
train_ids = set(multi_cam_ids[:n_train])
eval_ids = set(multi_cam_ids[n_train:])

cam_names = sorted({cam for cam_map in all_crops.values() for cam in cam_map})
cam2id = {cam_name: idx for idx, cam_name in enumerate(cam_names)}

train_data = []
query_data = []
gallery_data = []

train_pid_set = sorted(train_ids)
pid2label = {tid: idx for idx, tid in enumerate(train_pid_set)}
for tid in sorted(train_ids):
    label = pid2label[tid]
    for cam_name, paths in all_crops[tid].items():
        camid = cam2id[cam_name]
        for path in paths:
            train_data.append((path, label, camid))

eval_pid2label = {tid: idx for idx, tid in enumerate(sorted(eval_ids))}
for tid in sorted(eval_ids):
    pid = eval_pid2label[tid]
    for cam_name, paths in all_crops[tid].items():
        if not paths:
            continue
        camid = cam2id[cam_name]
        query_index = rng.randint(0, len(paths))
        query_data.append((paths[query_index], pid, camid))
        for index, path in enumerate(paths):
            if index != query_index:
                gallery_data.append((path, pid, camid))

distractor_pid = len(eval_ids)
for tid in sorted(single_cam_ids):
    for cam_name, paths in all_crops[tid].items():
        camid = cam2id[cam_name]
        for path in paths:
            gallery_data.append((path, distractor_pid, camid))
    distractor_pid += 1

num_classes = max(len(train_pid_set), 1)
num_cameras = max(len(cam_names), 1)
CAN_EVAL = len(query_data) > 0 and len(gallery_data) > 0

print(f"Train: {len(train_data)} images | IDs: {num_classes} | cameras: {num_cameras}")
print(f"Query: {len(query_data)} images")
print(f"Gallery: {len(gallery_data)} images")
print(f"Evaluation possible: {CAN_EVAL}")

train_tf = T.Compose([
    T.Resize((IMG_SIZE + 16, IMG_SIZE + 16), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(10),
    T.RandomCrop((IMG_SIZE, IMG_SIZE)),
    T.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.1, hue=0.0),
    T.ToTensor(),
    T.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    T.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value="random"),
])

test_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])

class ReIDDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        path, pid, cam = self.data[index]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, pid, cam, path

class PKSampler(Sampler):
    def __init__(self, data_source, p=PIDS_PER_BATCH, k=INSTANCES_PER_PID):
        self.data_source = data_source
        self.p = p
        self.k = k
        self.pid_to_indices = defaultdict(list)
        for index, (_, pid, _) in enumerate(data_source):
            self.pid_to_indices[pid].append(index)
        self.pids = list(self.pid_to_indices.keys())
        self.num_samples = max(1, len(self.pids)) * self.k

    def __iter__(self):
        pid_order = np.random.permutation(self.pids)
        batch_indices = []
        for pid in pid_order:
            indices = self.pid_to_indices[pid]
            replace = len(indices) < self.k
            chosen = np.random.choice(indices, self.k, replace=replace)
            batch_indices.extend(chosen.tolist())
        return iter(batch_indices)

    def __len__(self):
        return self.num_samples

train_ds = ReIDDataset(train_data, transform=train_tf)
query_ds = ReIDDataset(query_data, transform=test_tf)
gallery_ds = ReIDDataset(gallery_data, transform=test_tf)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=PKSampler(train_data),
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)
query_loader = DataLoader(query_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
gallery_loader = DataLoader(gallery_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Query batches: {len(query_loader)}")
print(f"Gallery batches: {len(gallery_loader)}")

In [ ]:
BACKBONE_ALIASES = {
	"vit_large_patch16_224_TransReID": [
		"vit_large_patch16_224.augreg_in21k_ft_in1k",
		"vit_large_patch16_224.augreg_in21k",
		"vit_large_patch16_224",
	],
	"vit_base_patch16_224_TransReID": [
		"vit_base_patch16_224.augreg_in21k_ft_in1k",
		"vit_base_patch16_224.augreg_in21k",
		"vit_base_patch16_224",
	],
}

BACKBONE_ALIASES["vit_large_patch14_dinov2"] = [
	"vit_large_patch14_dinov2.lvd142m",
	"vit_large_patch14_dinov2",
]

def resolve_backbone_name(vit_model: str) -> str:
	candidates = BACKBONE_ALIASES.get(vit_model, [vit_model])
	fallback = BACKBONE_ALIASES["vit_base_patch16_224_TransReID"]
	if vit_model == "vit_large_patch16_224_TransReID":
		candidates = candidates + fallback
	last_error = None
	for candidate in candidates:
		try:
			probe = timm.create_model(candidate, pretrained=False, num_classes=0, img_size=IMG_SIZE)
			del probe
			if candidate not in BACKBONE_ALIASES.get(vit_model, []):
				print(f"[WARN] Falling back to backbone alias {candidate} for {vit_model}")
			return candidate
		except Exception as error:
			last_error = error
	raise RuntimeError(f"Could not resolve a timm backbone for {vit_model}: {last_error}")

class TransReID(nn.Module):
	def __init__(
		self,
		num_classes: int,
		num_cameras: int = 0,
		embed_dim: int = 768,
		vit_model: str = "vit_base_patch16_224_TransReID",
		pretrained: bool = True,
		sie_camera: bool = True,
		jpm: bool = True,
		img_size: int | tuple[int, int] = 224,
		stride_size: int = 16,
	):
		super().__init__()
		self.sie_camera = sie_camera and num_cameras > 0
		self.jpm = jpm
		self.stride_size = stride_size
		self.requested_vit_model = vit_model
		self.timm_backbone = resolve_backbone_name(vit_model)
		self.vit = timm.create_model(
			self.timm_backbone,
			pretrained=pretrained,
			num_classes=0,
			img_size=img_size,
		)
		if hasattr(self.vit, "set_grad_checkpointing"):
			self.vit.set_grad_checkpointing(enable=True)
			print("[INFO] Gradient checkpointing enabled for ViT backbone")
		self.vit_dim = self.vit.embed_dim
		self.num_blocks = len(self.vit.blocks)

		if self.sie_camera:
			self.sie_embed = nn.Parameter(torch.zeros(num_cameras, self.vit_dim))
			nn.init.trunc_normal_(self.sie_embed, std=0.02)

		self.bn = nn.BatchNorm1d(self.vit_dim)
		self.bn.bias.requires_grad_(False)
		self.proj = nn.Linear(self.vit_dim, embed_dim, bias=False) if embed_dim != self.vit_dim else nn.Identity()
		self.cls_head = nn.Linear(embed_dim, num_classes, bias=False)
		if isinstance(self.proj, nn.Linear):
			nn.init.kaiming_normal_(self.proj.weight, mode="fan_out")
		nn.init.normal_(self.cls_head.weight, std=0.001)

		if self.jpm:
			self.bn_jpm = nn.BatchNorm1d(self.vit_dim)
			self.bn_jpm.bias.requires_grad_(False)
			self.jpm_cls = nn.Linear(self.vit_dim, num_classes, bias=False)
			nn.init.normal_(self.jpm_cls.weight, std=0.001)

		print(
			f"TransReID backbone={self.requested_vit_model} -> {self.timm_backbone} | "
			f"vit_dim={self.vit_dim} | embed_dim={embed_dim} | cameras={num_cameras}"
		)

	def forward(self, x: torch.Tensor, cam_ids: torch.Tensor | None = None):
		batch_size = x.shape[0]
		rot_pos_embed = None

		x = self.vit.patch_embed(x)
		if hasattr(self.vit, "_pos_embed"):
			pos_result = self.vit._pos_embed(x)
			if isinstance(pos_result, tuple):
				x, rot_pos_embed = pos_result
			else:
				x = pos_result
		else:
			cls_token = self.vit.cls_token.expand(batch_size, -1, -1)
			x = torch.cat([cls_token, x], dim=1) + self.vit.pos_embed
			if hasattr(self.vit, "pos_drop"):
				x = self.vit.pos_drop(x)

		if self.sie_camera and cam_ids is not None:
			# only add SIE to patch tokens, not cls token
			x[:, 1:] = x[:, 1:] + self.sie_embed[cam_ids].unsqueeze(1)

		if hasattr(self.vit, "patch_drop"):
			x = self.vit.patch_drop(x)
		if hasattr(self.vit, "norm_pre"):
			x = self.vit.norm_pre(x)

		for block in self.vit.blocks:
			if rot_pos_embed is not None:
				x = block(x, rope=rot_pos_embed)
			else:
				x = block(x)
		x = self.vit.norm(x)

		global_feat = x[:, 0]
		bn_feat = self.bn(global_feat)
		proj_feat = self.proj(bn_feat)

		if self.training:
			cls_logits = self.cls_head(proj_feat)
			if self.jpm:
				patches = x[:, 1:]
				shuffle_index = torch.randperm(patches.size(1), device=x.device)
				shuffled = patches[:, shuffle_index]
				midpoint = shuffled.size(1) // 2
				jpm_feat = (shuffled[:, :midpoint].mean(1) + shuffled[:, midpoint:].mean(1)) / 2
				jpm_logits = self.jpm_cls(self.bn_jpm(jpm_feat))
				return cls_logits, proj_feat, jpm_logits
			return cls_logits, proj_feat

		return F.normalize(proj_feat, p=2, dim=1)

	def get_llrd_param_groups(self, backbone_lr: float, head_lr: float, decay: float = 0.75):
		groups = {}
		for name, parameter in self.named_parameters():
			if not parameter.requires_grad:
				continue
			if name.startswith("vit."):
				if "blocks." in name:
					block_index = int(name.split("blocks.")[1].split(".")[0])
					depth = block_index + 1
				elif any(token in name for token in ("patch_embed", "cls_token", "pos_embed", "norm_pre")):
					depth = 0
				else:
					depth = self.num_blocks + 1
				scale = decay ** (self.num_blocks + 1 - depth)
				lr = backbone_lr * scale
				group_key = f"backbone_{depth}"
			else:
				lr = head_lr
				group_key = "head"
			groups.setdefault(group_key, {"params": [], "lr": lr})["params"].append(parameter)
		return sorted(groups.values(), key=lambda item: item["lr"])

model = TransReID(
	num_classes=num_classes,
	num_cameras=num_cameras,
	embed_dim=1024,
	vit_model=VIT_MODEL,
	pretrained=True,
	sie_camera=True,
	jpm=True,
	img_size=IMG_SIZE,
	stride_size=STRIDE_SIZE,
).to(DEVICE)

if torch.cuda.device_count() > 1:
	print(f"Using DataParallel across {torch.cuda.device_count()} GPUs")
	model = torch.nn.DataParallel(model)

print(f"Parameters: {sum(parameter.numel() for parameter in model.parameters()):,}")

In [ ]:
class CrossEntropyLabelSmooth(nn.Module):
	def __init__(self, num_classes: int, epsilon: float = 0.1):
		super().__init__()
		self.num_classes = num_classes
		self.epsilon = epsilon

	def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
		log_probs = F.log_softmax(inputs, dim=1)
		target_probs = F.one_hot(targets, num_classes=self.num_classes).float()
		target_probs = (1.0 - self.epsilon) * target_probs + self.epsilon / self.num_classes
		return (-target_probs * log_probs).sum(dim=1).mean()

class CenterLoss(nn.Module):
	def __init__(self, num_classes: int, feat_dim: int):
		super().__init__()
		self.centers = nn.Parameter(torch.randn(num_classes, feat_dim))

	def forward(self, features: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
		labels = labels.long()
		batch_centers = self.centers.index_select(0, labels)
		return 0.5 * (features - batch_centers).pow(2).sum(dim=1).mean()

def batch_hard_triplet_loss(features: torch.Tensor, labels: torch.Tensor, margin: float = 0.3) -> torch.Tensor:
	normalized = F.normalize(features, p=2, dim=1)
	distance = torch.cdist(normalized, normalized, p=2)
	labels = labels.view(-1, 1)
	positive_mask = labels.eq(labels.t())
	eye_mask = torch.eye(labels.size(0), dtype=torch.bool, device=labels.device)
	positive_mask = positive_mask & ~eye_mask
	negative_mask = ~labels.eq(labels.t())

	if positive_mask.sum() == 0 or negative_mask.sum() == 0:
		return features.sum() * 0.0

	hardest_positive = distance.masked_fill(~positive_mask, float("-inf")).max(dim=1).values
	hardest_negative = distance.masked_fill(~negative_mask, float("inf")).min(dim=1).values
	valid = torch.isfinite(hardest_positive) & torch.isfinite(hardest_negative)
	if valid.sum() == 0:
		return features.sum() * 0.0
	return F.relu(hardest_positive[valid] - hardest_negative[valid] + margin).mean()

ce_loss = CrossEntropyLabelSmooth(num_classes=num_classes, epsilon=0.05).to(DEVICE)
center_loss_fn = CenterLoss(num_classes=num_classes, feat_dim=1024).to(DEVICE)

raw_model = model.module if hasattr(model, "module") else model
optimizer = torch.optim.AdamW(
	raw_model.get_llrd_param_groups(BACKBONE_LR, HEAD_LR, decay=0.75),
	weight_decay=WEIGHT_DECAY,
)
center_optimizer = torch.optim.SGD(center_loss_fn.parameters(), lr=0.5)
base_lrs = [group["lr"] for group in optimizer.param_groups]
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")

def set_epoch_lrs(epoch_index: int) -> None:
	if epoch_index < WARMUP_EPOCHS:
		scale = float(epoch_index + 1) / float(max(WARMUP_EPOCHS, 1))
	else:
		progress = float(epoch_index - WARMUP_EPOCHS + 1) / float(max(NUM_EPOCHS - WARMUP_EPOCHS, 1))
		scale = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
	for group, base_lr in zip(optimizer.param_groups, base_lrs):
		group["lr"] = base_lr * scale

history = {"loss": [], "train_acc": []}
best_train_acc = -1.0
best_epoch = -1
final_train_acc = 0.0
eval_metrics = {}

if len(train_loader) == 0:
	raise RuntimeError("Train loader is empty; crop extraction or splitting failed")

print("=" * 72)
print(f"Training TransReID DINOv2 ViT-L/14 on CityFlowV2 | IDs={num_classes} | cameras={num_cameras}")
print(f"LRs: backbone={BACKBONE_LR:.2e}, head={HEAD_LR:.2e} | epochs={NUM_EPOCHS}")
print("Losses: ID + batch-hard triplet + center")
print("=" * 72)

for epoch in range(1, NUM_EPOCHS + 1):
	set_epoch_lrs(epoch - 1)
	model.train()
	running_loss = 0.0
	total_seen = 0
	total_correct = 0
	total_batches = 0
	use_center = (epoch - 1) >= CENTER_START

	for images, pids, cams, _ in train_loader:
		images = images.to(DEVICE, non_blocking=True)
		pids = pids.to(DEVICE, non_blocking=True).long()
		cams = cams.to(DEVICE, non_blocking=True).long()

		optimizer.zero_grad(set_to_none=True)
		if use_center:
			center_optimizer.zero_grad(set_to_none=True)

		with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
			outputs = model(images, cams)  # positional so DataParallel scatters cam_ids
			if len(outputs) == 3:
				cls_logits, features, jpm_logits = outputs
				id_loss = ce_loss(cls_logits, pids) + 0.5 * ce_loss(jpm_logits, pids)
			else:
				cls_logits, features = outputs
				id_loss = ce_loss(cls_logits, pids)
			triplet_loss = batch_hard_triplet_loss(features, pids, margin=MARGIN)
			total_loss = id_loss + triplet_loss
			if use_center:
				center_loss = center_loss_fn(features.float(), pids)
				total_loss = total_loss + CENTER_LOSS_WEIGHT * center_loss

		scaler.scale(total_loss).backward()
		scaler.unscale_(optimizer)
		torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=5.0)
		if use_center:
			scaler.unscale_(center_optimizer)
			torch.nn.utils.clip_grad_norm_(center_loss_fn.parameters(), 10.0)
		scaler.step(optimizer)
		if use_center:
			scaler.step(center_optimizer)
		scaler.update()

		predictions = cls_logits.argmax(dim=1)
		total_correct += int((predictions == pids).sum().item())
		total_seen += int(pids.numel())
		running_loss += float(total_loss.detach().item())
		total_batches += 1

	epoch_loss = running_loss / max(total_batches, 1)
	train_acc = total_correct / max(total_seen, 1)
	history["loss"].append(epoch_loss)
	history["train_acc"].append(train_acc)
	final_train_acc = train_acc

	if train_acc > best_train_acc:
		best_train_acc = train_acc
		best_epoch = epoch
		torch.save(raw_model.state_dict(), BEST_MODEL_PATH)

	if epoch == 1 or epoch % 10 == 0 or epoch == NUM_EPOCHS:
		print(
			f"Epoch {epoch:03d}: loss={epoch_loss:.4f}, acc={train_acc:.4f}, "
			f"head_lr={optimizer.param_groups[-1]['lr']:.2e}"
		)

print(f"Training complete | Best epoch: {best_epoch} | Final train acc: {final_train_acc:.4f}")
print(f"Best training accuracy: {best_train_acc:.4f}")

In [ ]:
@torch.no_grad()
def extract_features(model: nn.Module, loader: DataLoader, device: str = "cuda", flip: bool = True, pass_cams: bool = False):
	model.eval()
	features, pids, cams = [], [], []
	for images, pid, cam, _ in loader:
		images = images.to(device)
		kwargs = {"cam_ids": cam.to(device).long()} if pass_cams else {}
		feats = model(images, **kwargs)
		if isinstance(feats, (tuple, list)):
			feats = feats[-1]
		if flip:
			flipped = model(torch.flip(images, [3]), **kwargs)
			if isinstance(flipped, (tuple, list)):
				flipped = flipped[-1]
			feats = (feats + flipped) / 2.0
		feats = F.normalize(feats, p=2, dim=1)
		features.append(feats.cpu().numpy())
		pids.append(pid.numpy())
		cams.append(cam.numpy())
	if not features:
		return (
			np.zeros((0, 1024), dtype=np.float32),
			np.zeros((0,), dtype=np.int64),
			np.zeros((0,), dtype=np.int64),
		)
	return np.concatenate(features), np.concatenate(pids), np.concatenate(cams)

def eval_market1501_chunked(
	query_features: np.ndarray,
	gallery_features: np.ndarray,
	q_pids: np.ndarray,
	g_pids: np.ndarray,
	q_cams: np.ndarray,
	g_cams: np.ndarray,
	chunk_size: int = 1024,
	max_rank: int = 50,
):
	if query_features.shape[0] == 0 or gallery_features.shape[0] == 0:
		return 0.0, np.zeros(max_rank, dtype=np.float32)

	num_gallery = gallery_features.shape[0]
	max_rank = min(max_rank, num_gallery)
	gallery_features_t = gallery_features.T
	all_cmc = []
	all_ap = []

	for start in range(0, query_features.shape[0], chunk_size):
		end = min(start + chunk_size, query_features.shape[0])
		q_chunk = query_features[start:end]
		q_pid_chunk = q_pids[start:end]
		q_cam_chunk = q_cams[start:end]

		distances = 1.0 - q_chunk @ gallery_features_t
		indices = np.argsort(distances, axis=1)
		matches = (g_pids[indices] == q_pid_chunk[:, None]).astype(np.int32)

		for local_index in range(end - start):
			q_pid = q_pid_chunk[local_index]
			q_cam = q_cam_chunk[local_index]
			order = indices[local_index]
			remove = (g_pids[order] == q_pid) & (g_cams[order] == q_cam)
			keep = ~remove
			raw_cmc = matches[local_index][keep]
			if raw_cmc.sum() == 0:
				continue

			cmc = raw_cmc.cumsum()
			cmc[cmc > 1] = 1
			cmc = cmc[:max_rank]
			if cmc.shape[0] < max_rank:
				pad_value = int(cmc[-1]) if cmc.shape[0] else 0
				cmc = np.pad(cmc, (0, max_rank - cmc.shape[0]), constant_values=pad_value)
			all_cmc.append(cmc)

			num_rel = raw_cmc.sum()
			tmp_cmc = raw_cmc.cumsum()
			precision = tmp_cmc / (np.arange(len(tmp_cmc)) + 1.0)
			all_ap.append(float((precision * raw_cmc).sum() / num_rel))

	if not all_cmc:
		return 0.0, np.zeros(max_rank, dtype=np.float32)

	return float(np.mean(all_ap)), np.asarray(all_cmc, dtype=np.float32).mean(axis=0)

if BEST_MODEL_PATH.exists():
	best_state = torch.load(BEST_MODEL_PATH, map_location="cpu", weights_only=False)
	raw_model.load_state_dict(best_state, strict=False)

if CAN_EVAL:
	query_features, query_pids, query_cams = extract_features(raw_model, query_loader, DEVICE, pass_cams=True)
	gallery_features, gallery_pids, gallery_cams = extract_features(raw_model, gallery_loader, DEVICE, pass_cams=True)
	mean_ap, cmc = eval_market1501_chunked(
		query_features,
		gallery_features,
		query_pids,
		gallery_pids,
		query_cams,
		gallery_cams,
		chunk_size=EVAL_CHUNK_SIZE,
	)
	eval_metrics = {
		"mAP": float(mean_ap),
		"rank1": float(cmc[0]) if len(cmc) else 0.0,
		"num_query": int(len(query_pids)),
		"num_gallery": int(len(gallery_pids)),
	}
	print(f"Evaluation complete | mAP={eval_metrics['mAP']:.4f} | Rank-1={eval_metrics['rank1']:.4f}")
else:
	eval_metrics = {
		"mAP": None,
		"rank1": None,
		"num_query": 0,
		"num_gallery": 0,
	}
	print("No query/gallery split available; skipping retrieval evaluation and keeping train metrics only.")

In [ ]:
state_to_save = raw_model.state_dict()
if BEST_MODEL_PATH.exists():
	state_to_save = torch.load(BEST_MODEL_PATH, map_location="cpu", weights_only=False)

torch.save(state_to_save, FINAL_MODEL_PATH)

summary = {
	"backbone": VIT_MODEL,
	"resolved_backbone": getattr(raw_model, "timm_backbone", None),
	"embed_dim": 1024,
	"img_size": IMG_SIZE,
	"stride_size": STRIDE_SIZE,
	"batch_size": BATCH_SIZE,
	"num_epochs": NUM_EPOCHS,
	"best_epoch": best_epoch,
	"best_train_acc": float(best_train_acc),
	"final_train_acc": float(final_train_acc),
	"evaluation": eval_metrics,
	"num_classes": num_classes,
	"num_cameras": num_cameras,
}
with SUMMARY_PATH.open("w", encoding="utf-8") as handle:
	json.dump(summary, handle, ensure_ascii=True, indent=1)

print(f"Saved to {FINAL_MODEL_PATH}")
print(f"Summary saved to {SUMMARY_PATH}")